# Time Signature Bias in AI Music Generation

Two research questions:
- **RQ1 (User-experience bias)**: Do cultural prompts ("waltz") yield more accurate time signatures than formal ones ("3/4 time")?
- **RQ2 (Cultural bias)**: Do models collapse non-4/4 time signatures to 4/4?

In [ ]:
import os
import glob
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import fisher_exact

warnings.filterwarnings("ignore")
os.makedirs("figures", exist_ok=True)

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 12
sns.set_theme(style="whitegrid")

TIME_SIGS = ["2/4", "3/4", "4/4", "5/4", "6/8"]

In [ ]:
# Load all results.csv from audio subfolders (one per model)
dfs = []
for csv_path in glob.glob("audio/*/results.csv"):
    dfs.append(pd.read_csv(csv_path))

if not dfs:
    print("No results.csv found — run analyze_folder.py first.")
else:
    data = pd.concat(dfs, ignore_index=True)
    print(f"Loaded {len(data)} files | models: {data['model'].unique().tolist()}")
    display(data.head())

## Confidence Distribution & Free Tempo Threshold

Low confidence = madmom couldn't find a stable beat → likely free tempo output, not a real time signature.
Plot the distribution first, then set the threshold.

In [ ]:
fig, axes = plt.subplots(1, len(data["model"].unique()), figsize=(6 * len(data["model"].unique()), 4))
if len(data["model"].unique()) == 1:
    axes = [axes]

for ax, model in zip(axes, data["model"].unique()):
    df_m = data[data["model"] == model]
    for ts in TIME_SIGS:
        df_m[df_m["desired_time_sig"] == ts]["confidence"].hist(
            ax=ax, bins=20, alpha=0.5, label=ts
        )
    ax.set_title(model)
    ax.set_xlabel("Confidence")
    ax.set_ylabel("Count")
    ax.legend(fontsize=9)

plt.suptitle("Confidence Score Distribution by Desired Time Signature", fontsize=13)
plt.tight_layout()
plt.show()

# ── Set threshold here based on the distribution above ──────────────────────
FREE_TEMPO_THRESHOLD = 0.3

data["is_free_tempo"] = data["confidence"] < FREE_TEMPO_THRESHOLD

n_free = data["is_free_tempo"].sum()
print(f"Free tempo flagged: {n_free}/{len(data)} ({n_free/len(data)*100:.1f}%)")
print(data.groupby(["model", "desired_time_sig"])["is_free_tempo"].mean().round(2).to_string())

# All subsequent analysis uses only metered songs
data_free = data[data["is_free_tempo"]].copy()
data = data[~data["is_free_tempo"]].copy()
print(f"\nProceeding with {len(data)} metered songs.")

## Overview: Overall Accuracy

In [ ]:
# Overall accuracy per model
summary = (
    data.groupby("model")["correct"]
    .agg(correct="sum", total="count", accuracy="mean")
    .round(3)
)
display(summary)

# Accuracy breakdown: model × desired time signature
acc_table = (
    data.groupby(["model", "desired_time_sig"])["correct"]
    .mean()
    .unstack("desired_time_sig")
    .reindex(columns=TIME_SIGS)
    .round(3)
)
display(acc_table)

## Confusion Matrix

Each cell = number of songs with that desired→actual combination.
Diagonal = correct. A bright 4/4 column = cultural bias.

In [ ]:
models = data["model"].unique()
fig, axes = plt.subplots(1, len(models), figsize=(6 * len(models), 5))
if len(models) == 1:
    axes = [axes]

for ax, model in zip(axes, models):
    cm = (
        pd.crosstab(data[data["model"] == model]["desired_time_sig"],
                    data[data["model"] == model]["actual_time_sig"])
        .reindex(index=TIME_SIGS, columns=TIME_SIGS, fill_value=0)
    )
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                linewidths=0.5, cbar=False, square=True)
    ax.set_title(f"{model}", fontsize=13)
    ax.set_xlabel("Actual")
    ax.set_ylabel("Desired")

plt.suptitle("Confusion Matrix: Desired vs Actual Time Signature", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("figures/confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

## RQ1: User-Experience Bias — Cultural vs Formal Prompts

If cultural accuracy > formal accuracy, musically trained users are disadvantaged.

In [ ]:
acc_by_type = (
    data.groupby(["model", "desired_time_sig", "prompt_type"])["correct"]
    .mean()
    .reset_index()
)

g = sns.catplot(
    data=acc_by_type,
    x="desired_time_sig", y="correct", hue="prompt_type",
    col="model", kind="bar",
    palette={"cultural": "#4C72B0", "formal": "#DD8452"},
    height=5, aspect=1.1, order=TIME_SIGS,
)
g.set_axis_labels("Time Signature", "Accuracy")
g.set_titles("{col_name}")
g.add_legend(title="Prompt Type")
g.set(ylim=(0, 1))
plt.suptitle("RQ1: Accuracy by Prompt Type", y=1.02, fontsize=14)
plt.savefig("figures/rq1_prompt_type.png", dpi=150, bbox_inches="tight")
plt.show()

# Summary numbers
print(data.groupby(["model", "prompt_type"])["correct"].mean().round(3).to_string())

## RQ2: Cultural Bias — Collapse to 4/4

For songs where desired ≠ 4/4, what fraction was misdetected as 4/4?
Higher rate = stronger Western pop bias.

In [ ]:
non_44 = data[data["desired_time_sig"] != "4/4"].copy()
non_44["collapsed_to_44"] = non_44["actual_time_sig"] == "4/4"

collapse_rate = (
    non_44.groupby(["model", "desired_time_sig"])["collapsed_to_44"]
    .mean()
    .reset_index()
)

g = sns.catplot(
    data=collapse_rate,
    x="desired_time_sig", y="collapsed_to_44",
    col="model", kind="bar",
    color="#c44e52",
    height=5, aspect=1.1,
    order=[s for s in TIME_SIGS if s != "4/4"],
)
g.set_axis_labels("Desired Time Signature", "Collapse-to-4/4 Rate")
g.set_titles("{col_name}")
g.set(ylim=(0, 1))
plt.suptitle("RQ2: Rate of Collapsing to 4/4", y=1.02, fontsize=14)
plt.savefig("figures/rq2_collapse.png", dpi=150, bbox_inches="tight")
plt.show()

print(collapse_rate.to_string(index=False))

## Free Tempo Analysis

Songs flagged as free tempo — the model generated audio without a stable meter.
High rate for a time signature = the model couldn't handle it at all.

In [ ]:
free_rate = (
    pd.concat([data, data_free])
    .groupby(["model", "desired_time_sig"])["is_free_tempo"]
    .mean()
    .reset_index()
)

g = sns.catplot(
    data=free_rate,
    x="desired_time_sig", y="is_free_tempo",
    col="model", kind="bar",
    color="#888888",
    height=5, aspect=1.1, order=TIME_SIGS,
)
g.set_axis_labels("Desired Time Signature", "Free Tempo Rate")
g.set_titles("{col_name}")
g.set(ylim=(0, 1))
plt.suptitle("Rate of Free Tempo Output by Desired Time Signature", y=1.02, fontsize=14)
plt.savefig("figures/free_tempo_rate.png", dpi=150, bbox_inches="tight")
plt.show()

print(free_rate.to_string(index=False))

## Statistical Test: RQ1

Fisher's exact test (appropriate for small sample sizes) on cultural vs formal accuracy.
p < 0.05 = the difference is statistically significant.

In [ ]:
for model in data["model"].unique():
    df_m = data[data["model"] == model]
    ct = (
        pd.crosstab(df_m["prompt_type"], df_m["correct"])
        .reindex(index=["cultural", "formal"], columns=[False, True], fill_value=0)
    )
    _, p = fisher_exact(ct.values)

    c_acc = df_m[df_m["prompt_type"] == "cultural"]["correct"].mean()
    f_acc = df_m[df_m["prompt_type"] == "formal"]["correct"].mean()

    sig = "* significant" if p < 0.05 else "not significant"
    print(f"{model}:")
    print(f"  cultural accuracy : {c_acc:.3f}")
    print(f"  formal accuracy   : {f_acc:.3f}")
    print(f"  difference        : {c_acc - f_acc:+.3f}")
    print(f"  p-value           : {p:.4f}  ({sig})")
    print()

## Tempo Analysis

Does the model generate different tempos for different time signatures?
A model with cultural bias might generate 4/4-typical tempos even when asked for 5/4.

In [ ]:
fig, axes = plt.subplots(1, len(models), figsize=(6 * len(models), 5))
if len(models) == 1:
    axes = [axes]

for ax, model in zip(axes, models):
    df_m = data[data["model"] == model]
    groups = [df_m[df_m["desired_time_sig"] == ts]["tempo"].dropna() for ts in TIME_SIGS]
    ax.boxplot(groups, labels=TIME_SIGS, patch_artist=True,
               boxprops=dict(facecolor="#a8c8e8"))
    ax.set_title(model)
    ax.set_xlabel("Desired Time Signature")
    ax.set_ylabel("Detected Tempo (BPM)")

plt.suptitle("Tempo Distribution by Desired Time Signature", fontsize=14)
plt.tight_layout()
plt.savefig("figures/tempo.png", dpi=150, bbox_inches="tight")
plt.show()